In [44]:
import tensorflow as tf 
from tensorflow.keras.applications import VGG16
from tensorflow.keras.datasets import mnist
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
import numpy as np
import matplotlib as plt


In [45]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

x_train, y_train = x_train[:5000], y_train[:5000]
x_test, y_test = x_test[:1000], y_test[:1000]   

x_train = np.stack([x_train]* 3 , axis=-1)
x_test = np.stack([x_test]* 3 , axis=-1)

x_train = tf.image.resize(x_train, (224, 224)).numpy() /255.0
x_test = tf.image.resize(x_test, (224, 224)).numpy() /255.0

y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)


In [46]:
base_model = VGG16(weights = 'imagenet', include_top = False, input_shape = (224, 224, 3))

In [47]:
for layer in base_model.layers:
    layer.trainable = False
    

In [48]:
x = base_model.output
x = Flatten()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(10, activation='softmax')(x)
model = Model(inputs=base_model.input, outputs=output)

In [49]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
history = model.fit(x_train, y_train, epochs=5, batch_size=64, validation_split=0.2, verbose=1)

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'Test accuracy: {test_acc}')


In [51]:
for layer in base_model.layers[-4:]:
    layer.trainable = True

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
history_fine = model.fit(x_train, y_train, epochs=2, batch_size=64, validation_split=0.2, verbose=1)

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'Test accuracy after fine-tuning: {test_acc}')

In [ ]:
fig, axs = plt.subplots(2, 5, figsize=(12, 5))

indices = np.random.choice(len(x_test), 10 , replace=False)

for i, idx in enumerate(indices):
    ax = axs[i // 5, i % 5]
    pred = model.predict(x_test[idx: idx+1], verbose=0)
    pred_label = np.argmax(pred)
    true = np.argmax(y_test[idx])
    ax.imshow(x_test[idx][:,:,0], cmap='gray')
    ax.set_title(f'True: {true}, Pred: {pred}')
    ax.axis('off')

AttributeError: module 'matplotlib' has no attribute 'subplots'